In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


In [4]:
file_path = r"/content/electronics_product.csv"
df= pd.read_csv(file_path)

In [5]:
df.drop(columns=['Unnamed: 0', 'image', 'link'], inplace=True)

In [6]:
df.shape
df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.isnull().sum()

,0
name,0
main_category,0
sub_category,0
ratings,92
no_of_ratings,92
discount_price,473
actual_price,70


In [7]:
df['ratings'] = pd.to_numeric(df['ratings'], errors='coerce')
df['no_of_ratings'] = pd.to_numeric(df['no_of_ratings'], errors='coerce')
df.head()

,name,main_category,sub_category,ratings,no_of_ratings,discount_price,actual_price
0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,4.0,965.0,"₹10,999","₹18,999"
1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,4.3,NaN,"₹18,999","₹19,999"
2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,4.2,NaN,"₹1,999","₹2,299"
3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,4.1,NaN,"₹15,999","₹24,999"
4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,4.3,NaN,"₹18,999","₹19,999"


In [8]:
df.isna().sum()

,0
name,0
main_category,0
sub_category,0
ratings,110
no_of_ratings,4767
discount_price,473
actual_price,70


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score

# Filling missing values for columns we'll use for clustering
df['ratings'] = df['ratings'].fillna(df['ratings'].mean())
df['no_of_ratings'] = df['no_of_ratings'].fillna(0)

# Combine text features for a richer representation
df['combined_features'] = df['name'] + ' ' + df['main_category'] + ' ' + df['sub_category']
df['combined_features'] = df['combined_features'].fillna('')

# Vectorizing the text data
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

TF-IDF matrix shape: (9037, 5000)


In [10]:
# Splitting the vectorized data into train and test sets
X_train, X_test = train_test_split(tfidf_matrix, test_size=0.2, random_state=42)

print(f'Train set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Train set shape: (7229, 5000)
Test set shape: (1808, 5000)


In [11]:
# Applying KMeans Clustering
num_clusters = 10
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init='auto')
kmeans.fit(X_train)

# Predict clusters for the test set
test_labels = kmeans.predict(X_test)

print(f'Model trained with {num_clusters} clusters.')

Model trained with 10 clusters.


In [13]:
# Performance Metrics
# Silhouette Score measures how similar an object is to its own cluster compared to other clusters
score = silhouette_score(X_test, test_labels, sample_size=2000)
print(f"Silhouette Score on Test Set: {score:.4f}")

# Correctly mapping test labels to the original dataframe using .loc
_, test_indices = train_test_split(df.index, test_size=0.2, random_state=42)
df_test_results = df.loc[test_indices].copy()
df_test_results["cluster"] = test_labels

# Displaying the first 10 items and their assigned clusters
print("Sample results from the test set:")
display(df_test_results[["name", "cluster"]].head(10))

Silhouette Score on Test Set: 0.0224
Sample results from the test set:


,name,cluster
2057,AEIDESS Tempered Glass Screen Protector Compat...,1
1417,Ambrane Type-C to Micro USB OTG Adapter for 3A...,0
5081,"boAt Blaze Smart Watch with 1.75” HD Display, ...",7
2255,Wolpin Portable Travel Organizer Case for Earp...,8
6037,GM 3207 G-On Mini 3 Pin Extension Cord 1.5 Mtr...,8
8659,FriendZon 9H Nano Speedometer Screen Protector...,8
8828,HELLO ZONE Exclusive Rubber Matte Finish Soft ...,4
8340,MAYA BALLE® Camera Lens Protector Compatible f...,1
8820,TOUGH LEE Case Cover for Samsung Galaxy Buds 2...,4
4746,Prolet 20mm/Watch 4 Silicone Smart Watch Repla...,7


In [14]:
import numpy as np

print("Top terms per cluster:")
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf.get_feature_names_out()

for i in range(num_clusters):
    print(f"Cluster {i}:"),
    cluster_terms = [terms[ind] for ind in order_centroids[i, :10]]
    print(f" {cluster_terms}")

    # Show sample products from this cluster
    sample_products = df_test_results[df_test_results['cluster'] == i]['name'].head(3).values
    for prod in sample_products:
        print(f"   - {prod[:80]}...")
    print("-" * 30)

Top terms per cluster:
Cluster 0:
 ['usb', 'cable', 'type', 'charging', 'charger', 'fast', 'adapter', 'audio', 'tv', 'cameras']
   - Ambrane Type-C to Micro USB OTG Adapter for 3A Fast Charging and Data Sync- Comp...
   - D2Q Vivo Compatible Fast Charging USB Data Cable for Vivo Smartphones Supports H...
   - Amazon Brand - Solimo Fast Charging Braided Type C Data Cable Seam, Suitable For...
------------------------------
Cluster 1:
 ['edge', 'tempered', 'glass', 'screen', 'protector', 'coverage', 'easy', 'iphone', 'compatible', 'pro']
   - AEIDESS Tempered Glass Screen Protector Compatible for iPhone 11 with Edge to Ed...
   - MAYA BALLE® Camera Lens Protector Compatible for iPhone 13 Pro/iPhone 13 Pro Max...
   - CROSSVOLT Tempered Glass Screen Protector Compatible for Samsung Galaxy Watch 5 ...
------------------------------
Cluster 2:
 ['mouse', 'gaming', 'keyboard', 'dpi', 'wireless', 'optical', 'usb', 'wired', 'pad', 'tv']
   - Urbanmade Mouse Pad Large Office Table Accessories D

In [15]:
from sklearn.metrics import davies_bouldin_score, adjusted_rand_score
from sklearn.preprocessing import LabelEncoder

# 1. Davies-Bouldin Index (Lower is better)
db_index = davies_bouldin_score(X_test.toarray(), test_labels)
print(f"Davies-Bouldin Index: {db_index:.4f}")

# 2. Adjusted Rand Index (ARI)
# We need ground truth labels. Let's use 'main_category' as a proxy for ground truth
le = LabelEncoder()
true_labels_encoded = le.fit_transform(df.loc[test_indices, 'main_category'])

ari_score = adjusted_rand_score(true_labels_encoded, test_labels)
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")

print("\nInterpretation:")
print(f"- A DB Index of {db_index:.2f} suggests the clusters have some overlap.")
print(f"- An ARI of {ari_score:.2f} indicates the degree of agreement with the original main categories.")

Davies-Bouldin Index: 5.3353
Adjusted Rand Index (ARI): 0.0000

Interpretation:
- A DB Index of 5.34 suggests the clusters have some overlap.
- An ARI of 0.00 indicates the degree of agreement with the original main categories.


In [16]:
import pickle

# Save the model and the vectorizer
with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save the processed dataframe for lookups
df.to_pickle('product_data.pkl')

print("Model, Vectorizer, and Data successfully pickled!")

def get_recommendations(query, tfidf_model, kmeans_model, data, top_n=5):
    """
    Utility function for website integration.
    Takes a search query and returns the most relevant products from the assigned cluster.
    """
    # Vectorize the input query
    query_vector = tfidf_model.transform([query])

    # Predict the cluster
    cluster_id = kmeans_model.predict(query_vector)[0]

    # Filter data for that cluster
    recommendations = data[data['cluster_label'] == cluster_id].head(top_n)
    return recommendations[['name', 'main_category', 'actual_price']]

# Example usage for your backend:
# recommendations = get_recommendations('wireless headphones', tfidf, kmeans, df_test_results)
# print(recommendations)

Model, Vectorizer, and Data successfully pickled!
